# DAY1 실습 02 — 분포와 시각화 · 이상값 후보 — 정답

이 노트북은 **실습 문제만** 담고 있다. 개념과 기본 문법은 PPT 「DAY1 기술통계와 데이터 분포」에서 확인한다.
각 문제 아래의 `PPT p.번호`가 참고할 슬라이드이다. 빈 코드 셀에 답을 작성하고 실행한다.

In [ ]:
%pip install -q koreanize-matplotlib   # 한글 폰트 자동 설정. Colab은 세션마다 다시 설치한다

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
data_folder = "/content/drive/MyDrive/HD-AI/data"
csv_path = os.path.join(data_folder, "ai4i2020.csv")

df = pd.read_csv(csv_path)   # 원본
data = df.copy()             # 실습용 사본

In [ ]:
# [데이터 준비] 문제 1·2에 사용한다. 그대로 실행한다 (각 31개)
symmetric    = [10] + [20]*3 + [30]*7 + [40]*9 + [50]*7 + [60]*3 + [70]
right_skewed = [10]*12 + [15]*8 + [20]*4 + [30]*3 + [40]*2 + [65] + [90]
left_skewed  = [90]*12 + [85]*8 + [80]*4 + [70]*3 + [60]*2 + [35] + [10]
data1 = pd.DataFrame({"대칭": symmetric, "오른쪽 치우침": right_skewed, "왼쪽 치우침": left_skewed})

### 문제 1. 세 분포의 요약표
`PPT p.69`

세 분포의 평균, 중앙값, 표준편차, 왜도를 열로 갖는 표 `summary`를 만들어 소수 둘째 자리까지 출력하시오.

**확인** : 오른쪽 치우침 : 평균 20.81, 중앙값 15.0, 왜도 2.68

In [ ]:
summary = pd.DataFrame({"평균": data1.mean(), "중앙값": data1.median(), "표준편차": data1.std(), "왜도": data1.skew()})
summary.round(2)

### 문제 2. 같은 구간으로 히스토그램 비교
`PPT p.72, 69`

세 분포의 히스토그램을 0~100, 10 단위 구간으로 1행 3열에 그리시오.

In [ ]:
data1.hist(bins=range(0, 101, 10), figsize=(12, 3), layout=(1, 3), edgecolor="black")
plt.tight_layout()
plt.show()

### 문제 3. 회전속도 : bins 10 vs 40
`PPT p.73`

`Rotational speed [rpm]`의 히스토그램을 bins=10과 bins=40으로 각각 그리고(제목·축 이름 포함), 두 그래프의 차이를 주석으로 적으시오.

In [ ]:
speed = data["Rotational speed [rpm]"]
for k in [10, 40]:
    speed.hist(bins=k, edgecolor="white")
    plt.title(f"회전속도 : bins={k}"); plt.xlabel("회전속도(rpm)"); plt.ylabel("도수")
    plt.show()
# bins=40에서 봉우리 모양과 긴 오른쪽 꼬리가 더 자세히 보인다

### 문제 4. IQR 기준 이상값 후보
`PPT p.75`

회전속도의 Q1, Q3, IQR과 하한·상한(1.5×IQR)을 구하고, 경계 밖 값을 `iqr_mask`에 담아 후보 수를 세시오.

**확인** : 1139.5, 1895.5 / 418건

In [ ]:
q1, q3 = speed.quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
iqr_mask = (speed < lower) | (speed > upper)
print("하한·상한(rpm):", round(lower, 1), round(upper, 1))
print("IQR 기준 이상값 후보 수:", iqr_mask.sum())

### 문제 5. 회전속도 박스플롯
`PPT p.75, 76`

회전속도 박스플롯을 수염 1.5×IQR, 평균 표시 옵션으로 그리시오(제목·축 이름 포함).

In [ ]:
speed.plot.box(whis=1.5, showmeans=True)
plt.title("회전속도 박스플롯"); plt.ylabel("회전속도(rpm)")
plt.show()

### 문제 6. z-score 계산
`PPT p.89`

평균 50, 표준편차 10이라고 할 때 `[30, 50, 70]`의 z-score를 구하시오.

**확인** : [-2.0, 0.0, 2.0]

In [ ]:
values = pd.Series([30, 50, 70])
z_scores = (values - 50) / 10
print(z_scores.tolist())

### 문제 7. 회전속도의 z-score
`PPT p.89`

회전속도의 평균과 표준편차(ddof=1)로 각 값의 z-score를 계산해 `z_speed`에 담고 처음 5개를 출력하시오.

In [ ]:
z_speed = (speed - speed.mean()) / speed.std()
z_speed.head()

### 문제 8. |z| > 3 이상값 후보
`PPT p.91`

평균에서 표준편차 3배보다 멀리 떨어진 회전속도는 몇 건인가? 조건은 `z_mask`에 담고, 해당 행의 회전속도를 출력하시오.

**확인** : 164건

In [ ]:
z_mask = z_speed.abs() > 3
print("z-score 기준 이상값 후보:", z_mask.sum())
display(data.loc[z_mask, ["Rotational speed [rpm]"]])

### 문제 9. z-score 분포와 기준선
`PPT p.91`

`z_speed`의 히스토그램(bins=40)을 그리고 z = −3, +3 위치에 빨간 점선을 그으시오.

In [ ]:
z_speed.hist(bins=40, figsize=(9, 4))
plt.axvline(-3, color="red", linestyle="--")
plt.axvline(3, color="red", linestyle="--")
plt.title("회전속도 Z-score 분포"); plt.xlabel("Z-score"); plt.ylabel("빈도")
plt.show()

### 문제 10. 왜도와 초과첨도
`PPT p.94, 92, 93`

회전속도와 토크의 왜도 및 초과첨도는 각각 얼마인가?

**확인** : 회전속도 1.993 / 7.393, 토크 −0.010 / −0.013

In [ ]:
cols = ["Rotational speed [rpm]", "Torque [Nm]"]
pd.DataFrame({"왜도": data[cols].skew(), "초과첨도": data[cols].kurt()}).round(3)

### 문제 11. 히스토그램으로 모양 확인
`PPT p.94`

회전속도와 토크의 히스토그램(bins=30)을 1행 2열로 나란히 그리고, 문제 10의 숫자와 모양을 비교해 주석으로 적으시오.

In [ ]:
data[["Rotational speed [rpm]", "Torque [Nm]"]].hist(bins=30, figsize=(12, 4), layout=(1, 2), edgecolor="black")
plt.tight_layout(); plt.show()
# 회전속도 : 오른쪽 꼬리가 길다(왜도 +) / 토크 : 좌우 대칭에 가깝다(왜도 ≈ 0)

### 문제 12. IQR 기준 vs z-score 기준
`PPT p.95`

문제 4의 `iqr_mask`와 문제 8의 `z_mask`로 두 기준의 이상값 후보 수를 한 Series로 비교하고, 차이가 나는 이유를 주석으로 적으시오.

**확인** : 418 vs 164

In [ ]:
pd.Series({"IQR 기준 이상값 후보": int(iqr_mask.sum()), "z-score 기준 이상값 후보": int(z_mask.sum())})
# 치우친 분포에서는 큰 값들이 평균·표준편차를 키워 z 경계가 더 멀어진다

### 문제 13. 토크의 히스토그램과 박스플롯
`PPT p.76, 75`

`Torque [Nm]`의 히스토그램과 박스플롯을 그리고, 분포 모양과 이상값 후보를 확인하시오.

In [ ]:
torque = data["Torque [Nm]"]
torque.hist(bins=30, edgecolor="white"); plt.title("토크 히스토그램"); plt.xlabel("토크(Nm)"); plt.show()
torque.plot.box(showmeans=True); plt.title("토크 박스플롯"); plt.ylabel("토크(Nm)"); plt.show()
# 대칭에 가까운 종 모양, 양쪽 끝에 이상값 후보가 있다

### 문제 14. 토크의 IQR 경계와 후보 수
`PPT p.75, 95`

토크의 IQR 경계(하한·상한)와 이상값 후보 수를 계산하시오.

**확인** : 12.8, 67.2 / 69건

In [ ]:
q1, q3 = torque.quantile([0.25, 0.75]); iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print("하한·상한(Nm):", round(lower, 1), round(upper, 1))
print("이상값 후보 수:", ((torque < lower) | (torque > upper)).sum())